# YUNESA Academic Knowledge Graph Construction

Notebook ini membangun knowledge graph akademik dari data `papers`, `lecturers`, dan `paper_lecturers` di Supabase, lalu memperkaya node `Concept` memakai IEEE taxonomy/thesaurus yang sudah ada di folder ini.

Workflow notebook:

1. Load sample data paper dan dosen dari Supabase.
2. Load IEEE SKOS taxonomy/thesaurus sebagai controlled vocabulary.
3. Bangun backbone graph: `Dosen`, `Publikasi`, `Venue`, `Tahun`, `Keyword`.
4. Ekstrak `Concept` dari `title + tldr + abstract + keywords`.
5. Hubungkan publikasi ke concept sesuai ontology: `MEMBAHAS_TOPIK`, `MENGGUNAKAN_METODE`, `MENGGUNAKAN_MODEL`, `BERADA_PADA_DOMAIN`, `MENGGUNAKAN_DATASET`, `DIEVALUASI_DENGAN`.
6. Export hasil ke CSV, JSON node-link, dan GraphML untuk analisis lanjutan atau import Neo4j.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/rizkyyanuark/Tugas_Akhir.git'
REPO_DIR = Path('/content/Tugas_Akhir')

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR / 'notebooks' / 'build-graph')
    pip_packages = ['pandas', 'networkx', 'rdflib==6.3.1', 'supabase>=2.25.1', 'python-dotenv', 'pyvis']
    required_modules = ['pandas', 'networkx', 'rdflib', 'supabase', 'dotenv', 'pyvis']
else:
    pip_packages = []
    required_modules = ['pandas', 'networkx', 'rdflib', 'supabase', 'dotenv']

missing = [pkg for pkg in required_modules if importlib.util.find_spec(pkg) is None]
if missing and IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_packages])
elif missing:
    raise RuntimeError(
        'Notebook dependencies are missing: ' + ', '.join(missing) +
        '. Run: uv sync --project notebooks'
    )

BUILD_GRAPH_DIR = Path.cwd()
if not (BUILD_GRAPH_DIR / 'src').exists():
    BUILD_GRAPH_DIR = Path('notebooks/build-graph').resolve()

sys.path.insert(0, str(BUILD_GRAPH_DIR / 'src'))

from yunesa_academic_kg import (
    KGConfig,
    IeeeSemanticIndex,
    AcademicKGBuilder,
    export_graph_artifacts,
    fetch_supabase_sample,
    graph_to_frames,
    load_local_csv_sample,
    load_project_env,
)

config = KGConfig.default(sample_size=50)
load_project_env(config.project_root)
config


## 1. Load Data

Default loader mengambil data dari Supabase. Jika credential belum tersedia atau koneksi gagal, cell ini otomatis fallback ke CSV lokal di `notebooks/scraping/file_tabulars`.

In [ ]:
try:
    papers_df, lecturers_df, links_df = fetch_supabase_sample(sample_size=config.sample_size)
    data_source = 'supabase'
except Exception as exc:
    print(f'Supabase load failed, using local CSV fallback: {type(exc).__name__}: {exc}')
    papers_df, lecturers_df, links_df = load_local_csv_sample(
        config.project_root / 'notebooks' / 'scraping' / 'file_tabulars',
        sample_size=config.sample_size,
    )
    data_source = 'local_csv'

print('Data source:', data_source)
print('Papers:', len(papers_df))
print('Lecturers:', len(lecturers_df))
print('Paper-lecturer links:', len(links_df))
papers_df.head(3)


## 2. Load IEEE Semantic Index

Index ini membaca `ieee-thesaurus.ttl` dan `ieee-taxonomy.ttl`. IEEE labels dipakai untuk grounding concept, bukan sebagai satu-satunya sumber concept. Keyword mentah dan regex teknis tetap dipakai agar istilah spesifik seperti model, dataset, dan metric tidak hilang.

In [ ]:
ieee_index = IeeeSemanticIndex.from_files(
    config.thesaurus_path,
    config.taxonomy_path,
    max_terms=config.max_ieee_terms,
)
ieee_index.summary()


## 3. Build Knowledge Graph

Graph dibangun sebagai `networkx.MultiDiGraph`, sehingga satu pasang node bisa memiliki beberapa edge dengan relasi berbeda. Setiap edge concept menyimpan `source`, `match_type`, `matched_text`, `score`, dan `provenance`.

In [ ]:
builder = AcademicKGBuilder(ieee_index)
G = builder.build(
    papers_df=papers_df,
    lecturers_df=lecturers_df,
    links_df=links_df,
    max_concepts_per_paper=config.max_concepts_per_paper,
)

validation = builder.validate()
validation


## 4. Inspect Nodes and Edges

In [ ]:
nodes_df, edges_df = graph_to_frames(G)
display(nodes_df.groupby('node_type').size().sort_values(ascending=False).to_frame('count'))
display(edges_df.groupby('relation').size().sort_values(ascending=False).to_frame('count'))


In [ ]:
concept_edges = edges_df[edges_df['relation'].isin([
    'MEMBAHAS_TOPIK',
    'MENGGUNAKAN_METODE',
    'MENGGUNAKAN_MODEL',
    'BERADA_PADA_DOMAIN',
    'MENGGUNAKAN_DATASET',
    'DIEVALUASI_DENGAN',
])]
concept_edges[['source', 'target', 'relation', 'edge_source', 'match_type', 'matched_text', 'score']].head(20)


## 5. Export Artifacts

Output disimpan di `notebooks/build-graph/outputs/academic_kg/`:

- `academic_kg_nodes.csv`
- `academic_kg_edges.csv`
- `academic_kg_node_link.json`
- `academic_kg.graphml`
- `academic_kg_summary.json`

In [ ]:
artifacts = export_graph_artifacts(G, config.output_dir)
for name, path in artifacts.items():
    print(f'{name}: {path}')


## 6. Optional Visualization

Cell ini membuat HTML interaktif memakai PyVis. Jalankan hanya untuk sample kecil agar browser tetap ringan.

In [ ]:
# Optional: generate an interactive HTML graph for small samples.
from pyvis.network import Network

preview_path = config.output_dir / 'academic_kg_preview.html'
net = Network(height='760px', width='100%', bgcolor='#ffffff', directed=True, notebook=True)

color_by_type = {
    'Dosen': '#2f80ed',
    'Publikasi': '#27ae60',
    'Venue': '#8e44ad',
    'Tahun': '#7f8c8d',
    'Keyword': '#f39c12',
    'Concept': '#c0392b',
}

for node_id, data in G.nodes(data=True):
    node_type = data.get('node_type', 'Unknown')
    net.add_node(
        node_id,
        label=str(data.get('label', node_id))[:60],
        title=f"{node_type}: {data.get('label', node_id)}",
        color=color_by_type.get(node_type, '#95a5a6'),
    )

for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get('relation', ''), title=data.get('relation', ''))

net.show(str(preview_path))
preview_path
